In [ ]:
"""
postprocess_SPEI.ipynb
===============================
Post-processing of monthly SPEI/SPI-12 index rasters into WIA drought-impact
indicators.

Reads the per-month index GeoTIFFs produced by SPEI_impact.ipynb,
for the target calendar year and computes, per admin area, these
two indicator families for the given time period:

  POPULATION indicators
    1. Impacted population rate — % of admin pop living under the any-month-dry
       mask over the target year.
    2. Pop-weighted avg dry-months (total-population denominator).
    3. Pop-weighted avg dry-months (dry-affected pop denominator).

  AREA indicators
    4. Dry area-months — territorial drought persistence (km^2 * months).
    5. Avg dry-months per total area.
    6. Avg dry-months per dry area.

Drought
------------------------
The extraction step saves the SPEI/SPI-12 indexes.
The drought trigger (SPEI-12 <= -1.5) is applied here, per month per pixel,
from the raw index rasters.
  - dry-MONTH count (0..12)
  - any-month-dry (cum >= 1)
  - dry if SPEI <= DRY_THRESHOLD in that month
  - more dry-months = worse

Grid / transform
----------------
  TerraClimate rasters are EPSG:4326 (geographic).
  Original projection is kept (area indicators incorporate transformations).
  Downstream steps (WorldPop reprojection, admin rasterization, zonal stats)
  mirror GFM flood pipeline.

NOTE: postprocess_gfm_flood.ipynb and this one are good candidates for a
shared wia_utils module in a later refactor; kept self-contained for now.
"""

In [ ]:
# 0. IMPORTS
import json
import logging
import math
from datetime import date, datetime, timezone
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import pycountry
import pyproj
import rasterio
import rioxarray    # noqa: F401 – registers the .rio accessor
import xarray as xr
from rasterio.features import rasterize
from rasterio.warp import reproject, Resampling
from rasterio.transform import xy, rowcol

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

##### 1. CONFIGURATION — edit this section for your run

In [ ]:
# -- Folder structure --
base_dir = "./"
data_in_folder = base_dir + "data_in/"
data_in_shapes = base_dir + "data_in/shapes/"

# Drought index rasters produced by SPEI_impact.ipynb
drought_raster_dir = Path(data_in_folder + "Raster_Datasets/Drought_Index")

# WorldPop constrained population rasters
wpop_dir = data_in_folder + "WorldPop_rasters/G2_CN_POP_R25A_100m"

# Output folders
maps_hazards_folder = base_dir + "data_out/maps/hazards/drought/"
output_log_dir = base_dir + "data_out/logs/"
Path(maps_hazards_folder).mkdir(parents=True, exist_ok=True)
Path(output_log_dir).mkdir(parents=True, exist_ok=True)


In [ ]:
# -- Country / admin / year selection --
# NOTE: selected countries for WIA execution
wia_countries_exercise = [
    # "Kenya",
    "Mali",
    # "Benin",
    # "Lebanon",
    # "Togo",
    # "Afghanistan",
    # "Ukraine",
    # "Burkina Faso",
    # "Niger",
    # "Honduras",
    # "El Salvador",
    # "Cameroon",
    # "Central African Republic",
    # "Myanmar",
    # "South Sudan",
    # "Syria",
    # "Ethiopia",
    # "Congo, The Democratic Republic of the",
    # "Haiti",
    # "Somalia",
    # "Sudan",
    # "Yemen",
    # "Saint Vincent and the Grenadines",
    # "Grenada",
    # "Mozambique",
    # "State of Palestine",
]
country = wia_countries_exercise[0]

c_iso3 = (
    pycountry.countries.search_fuzzy(country)[0].alpha_3
    if country != "Niger"
    else pycountry.countries.search_fuzzy(country)[1].alpha_3
)

admin_level = "2"
# Population year and drought target year (last completed calendar year)
year = f"{date.today().year - 1}"
TARGET_YEAR = int(year)

pcode_col = f"adm{admin_level}_pcode"
name_col = f"adm{admin_level}_name"


In [ ]:
# -- Index selection --
# Primary drought index. SPI is left as a switchable input.
INDEX_KIND = "spei"     # "spi" #
INDEX_DIST = "pearson3" if INDEX_KIND == "spei" else "gamma"
INDEX_SCALE = 12

# Drought trigger: a pixel is "dry" in a month if index <= DRY_THRESHOLD.
# Standard severe-drought band (matches the exposure layer's reporting band).
DRY_THRESHOLD = -1.5

# NOTE: NaN handling doesn't look like an issue in TerraClimate
# The logic here implemented was used to conclude the above.
# -- NaN handling for the cumulative dry-month analysis --
# Drought NaN (undefined index for a pixel-month) is NOT treated as "not dry"
# We accumulate dry-months ignoring NaN months, but a pixel that is NaN
# in ALL 12 months is carried as nodata, not 0.
CUM_NODATA = 255        # uint8 sentinel for all-NaN pixels (0..12 are valid)
# Optional: require a minimum number of observed (non-NaN) months for a pixel to
# be eligible to count as "dry" in the any-month-dry mask. 1 = keep every pixel
# with at least one observed month ("maximize non-NaN" default).
MIN_VALID_MONTHS_FOR_DRY = 1


In [ ]:
# -- Reference grid / output resolution --
# Drought is limited by the HAZARD resolution (TerraClimate ~4.6 km)
# WorldPop is 100 m, so the output grid is TerraClimate's resolution

# -- Binarization / map settings --
NODATA_GREY = "Gray"
CSCALE_DROUGHT = "YlOrRd"          # more dry-months = worse = red
CSCALE_CUM_RASTER = "Viridis"      # cumulative dry-months raster
CSCALE_RASTER = "Jet"              # dry-months x pop raster

N_QUANTILES = 4


In [ ]:
# -- Label for filenames --
# Drought aligns to a calendar year, same as the population source
# Then, area and population indicators share ONE label.
drought_label = f"{c_iso3}_previous_adm{admin_level}_{TARGET_YEAR}"

# Index tag used in the raster filenames from the extraction notebook
# e.g. spei12_pearson3_{ISO3}_{YYYY-MM}.tif
index_tag = f"{INDEX_KIND}{INDEX_SCALE}_{INDEX_DIST}"

log.info(
    f"{country} ({c_iso3}) adm{admin_level} | index={index_tag} | "
    f"dry if <= {DRY_THRESHOLD} | target {TARGET_YEAR}"
)


##### 2. LOAD ADMIN BOUNDARIES

In [ ]:
geo_filename = f"geojson_{c_iso3}_previous_adm{admin_level}"
geojson_file = [
    f.name for f in Path(data_in_shapes).iterdir()
    if f.name.endswith(".zip") and geo_filename in f.name
]
if not geojson_file:
    raise FileNotFoundError(
        f"No admin file matching '{geo_filename}*.zip' in {data_in_shapes}"
    )

gdf = gpd.read_file(data_in_shapes + geojson_file[0]).to_crs("EPSG:4326")
geojson_dict = json.loads(gdf.to_json())
log.info(f"Loaded {len(gdf)} adm{admin_level} features for {c_iso3} from {geojson_file[0]}")


##### 3. REFERENCE GRID (TerraClimate native resolution)
The output grid uses TerraClimate's own resolution rather than 100 m.
We take the first index raster grid as the reference.
The remaining index rasters (by assumption) are already on this grid.
Only WorldPop is resampled onto it.

In [ ]:
# ── REFERENCE GRID = the native TerraClimate grid (EPSG:4326), no UTM ──
# We adopt the first index raster's OWN grid (transform, shape, CRS) as the
# reference. The monthly index rasters therefore need no reprojection: they are
# already on this grid. Only WorldPop is resampled onto it (Section 5).
_probe = drought_raster_dir / f"{index_tag}_{c_iso3}_{TARGET_YEAR}-01.tif"
if not _probe.exists():
    raise FileNotFoundError(
        f"Missing index raster {_probe.name} in {drought_raster_dir}. "
        f"Run SPEI_impact.ipynb for {c_iso3} {TARGET_YEAR}."
    )
with rasterio.open(_probe) as src0:
    ref_transform = src0.transform
    ref_crs = src0.crs                      # EPSG:4326 (TerraClimate native)
    ref_shape = (src0.height, src0.width)
    cell_deg_x = abs(src0.transform.a)      # lon cell size (deg)
    cell_deg_y = abs(src0.transform.e)      # lat cell size (deg)

log.info(f"Reference grid (native): {ref_shape[1]}x{ref_shape[0]} px, CRS={ref_crs}")
log.info(f"Cell size: {cell_deg_x:.6f} x {cell_deg_y:.6f} deg")

# ── Per-pixel area, in km^2, as a function of latitude (graticule cell area) ──
# A cell centred at lat with height dlat, width dlon (deg) on a sphere of radius R:
#   A = R^2 * dlon_rad * (sin(lat+dlat/2) - sin(lat-dlat/2))
# Longitude cells are uniform width in deg, but their ground area shrinks toward
# the poles, so area depends only on the row's latitude. We build a 2D area grid
# (one value per row, broadcast across columns) so downstream sums are exact.
_R_KM = 6371.0088
# latitude of each row centre from the transform:  y = e*row + f  (e is negative)
_rows = np.arange(ref_shape[0])
_row_lat = ref_transform.f + ref_transform.e * (_rows + 0.5)   # centre of each row
_lat1 = np.radians(_row_lat - cell_deg_y / 2.0)
_lat2 = np.radians(_row_lat + cell_deg_y / 2.0)
_row_area_km2 = (_R_KM ** 2) * np.radians(cell_deg_x) * (np.sin(_lat2) - np.sin(_lat1))
# 2D area grid (rows x cols); every column in a row shares the row's area
pixel_area_km2_grid = np.repeat(_row_area_km2[:, None], ref_shape[1], axis=1).astype(np.float64)
log.info(
    f"Pixel area varies by latitude: "
    f"{pixel_area_km2_grid.min():.3f}..{pixel_area_km2_grid.max():.3f} km^2 "
    f"(mean {pixel_area_km2_grid.mean():.3f})"
)


##### 4. DISCOVER & STACK INDEX RASTERS → DRY-MONTH COUNT
Each monthly index raster (EPSG:4326) is thresholded to a per-month
dry mask (index <= DRY_THRESHOLD), done in its native TerraClimate grid.
Then, it's summed across the 12 target-year months into a
cumulative dry-month count (0..12).

In [ ]:
# Expected filenames: {index_tag}_{ISO3}_{YYYY-MM}.tif for each target-year month
target_months = [f"{TARGET_YEAR}-{m:02d}" for m in range(1, 13)]

index_tifs = []
for ml in target_months:
    fp = drought_raster_dir / f"{index_tag}_{c_iso3}_{ml}.tif"
    if not fp.exists():
        raise FileNotFoundError(
            f"Missing index raster {fp.name} in {drought_raster_dir}. "
            f"Run SPEI_impact.ipynb for {c_iso3} {TARGET_YEAR}."
        )
    index_tifs.append((ml, fp))

log.info(f"Found {len(index_tifs)} monthly {index_tag} rasters for {c_iso3} {TARGET_YEAR}")


In [ ]:
# Stack: threshold each month on its NATIVE TerraClimate grid.
# NO reprojection is needed: (index <= DRY_THRESHOLD) and accumulate.

# A pixel that is NaN in ALL months is carried as nodata in the output raster,
# using CUM_NODATA distinct from a genuine 0 (observed, never dry).

# Two integer rasters result (both uint8, compact):
#   cumulative_dry_months : 0..12 dry-month count (nansum), exported with CUM_NODATA if all-NaN
#   valid_months          : 0..12 count of observed (non-NaN) months per pixel
cumulative_dry_months = np.zeros(ref_shape, dtype=np.uint8)
valid_months = np.zeros(ref_shape, dtype=np.uint8)
months_loaded = []

for month_label, tif_path in index_tifs:
    with rasterio.open(tif_path) as src:
        # sanity: every month must be on the SAME native grid as the reference
        if (src.width, src.height) != (ref_shape[1], ref_shape[0]):
            raise ValueError(
                f"{tif_path.name} grid {src.width}x{src.height} != reference "
                f"{ref_shape[1]}x{ref_shape[0]}. All months must share the grid."
            )
        src_data = src.read(1).astype(np.float32)
        src_nodata = src.nodata

    # Native-grid masks (no reprojection):
    #   valid_native : pixel has a real index value (not nodata/NaN)
    #   dry_native   : valid AND index <= threshold  -> 1
    if src_nodata is not None and not np.isnan(src_nodata):
        valid_native = src_data != src_nodata
    else:
        valid_native = ~np.isnan(src_data)
    dry_native = (valid_native & (src_data <= DRY_THRESHOLD)).astype(np.uint8)

    cumulative_dry_months += dry_native   # 0/1 per month, summed to 0..12
    valid_months += valid_native

    months_loaded.append(month_label)
    log.info(f"  Stacked {tif_path.name}: {int(dry_native.sum()):,} dry pixels (native grid)")

# All-NaN pixels: never observed in any month -> nodata, not 0
all_nan = valid_months == 0

# NOTE: log below should usually be 12 (both min and mean)
log.info(
    f"Validity: min={int(valid_months[~all_nan].min()) if (~all_nan).any() else 0}, "
    f"mean={valid_months[~all_nan].mean():.1f} observed months "
    f"(over non-all-NaN pixels)"
)


##### 4b. EXPORT CUMULATIVE DRY-MONTHS RASTER (GeoTIFF)

In [ ]:
def write_raster_cog(array, transform, crs, out_path, dtype="float32", nodata=None, blocksize=512):
    """Wrap a numpy grid as a georeferenced DataArray and write it as a COG.
    (Mirrors write_raster_cog in postprocess_gfm_flood.ipynb.)"""
    height, width = array.shape
    xs = xy(transform, np.zeros(width, dtype=int), np.arange(width))[0]
    ys = xy(transform, np.arange(height), np.zeros(height, dtype=int))[1]
    da = xr.DataArray(array.astype(dtype), dims=("y", "x"), coords={"y": ys, "x": xs})
    da = da.rio.write_crs(crs)
    if nodata is not None:
        da = da.rio.write_nodata(nodata)
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    da.rio.to_raster(
        str(out_path), driver="COG", compress="DEFLATE", dtype=dtype,
        overview_resampling="nearest", blocksize=blocksize,
    )
    log.info(f"GeoTIFF (COG) written: {out_path}")

# Encode the cumulative dry-month raster with a nodata sentinel
cumulative_dry_months_for_raster = cumulative_dry_months.copy()
cumulative_dry_months_for_raster[all_nan] = CUM_NODATA          # 255 where all-NaN
write_raster_cog(
    cumulative_dry_months_for_raster, ref_transform, ref_crs,
    maps_hazards_folder + f"{drought_label}_{index_tag}_cumulative_dry_months.tif",
    dtype="uint8", nodata=CUM_NODATA,   # <-- carries all-NaN as file nodata
)

# Companion validity raster — how many months were observed per pixel (QC)
# 0..12, no sentinel: 0 = "zero valid months" is real, not missing data
write_raster_cog(
    valid_months, ref_transform, ref_crs,
    maps_hazards_folder + f"{drought_label}_{index_tag}_valid_months.tif",
    dtype="uint8",  # no nodata= : every 0..12 is meaningful
)


##### 4c. CREATE ANY-MONTH-DRY BINARY MASK
Drought analogue of the any-day-flooded mask: any pixel dry in >= 1 month.

In [ ]:
# Any-month-dry mask: pixel dry in >= 1 observed month, AND meeting the minimum
# validity requirement. All-NaN pixels are excluded (they are nodata, not dry).
eligible = (valid_months >= MIN_VALID_MONTHS_FOR_DRY) & (~all_nan)
any_dry_mask = (eligible & (cumulative_dry_months > 0)).astype(np.uint8)
n_dry_pixels = int(any_dry_mask.sum())
log.info(
    f"Any-month-dry mask: {n_dry_pixels:,} pixels "
    f"(min validity = {MIN_VALID_MONTHS_FOR_DRY} month(s))"
)


##### 5. REPROJECT WORLDPOP TO REFERENCE GRID (downsample by SUM)
WorldPop (100 m) is **downsampled** onto the coarse TerraClimate-resolution grid.
Population is resampled with `sum` — each coarse cell receives the
sum of the WorldPop original pixels that fall in it (pop counts conserved)

In [ ]:
total_pop_rasters = [f.name for f in Path(wpop_dir).iterdir() if f.is_file()]
pop_raster_name = next(
    (f for f in total_pop_rasters if (c_iso3.lower() in f) and (f"_{year}_" in f)),
    None,
)
if pop_raster_name is None:
    raise FileNotFoundError(f"No WorldPop raster for {c_iso3} / {year} in {wpop_dir}")
pop_raster_path = str(Path(wpop_dir) / pop_raster_name)
log.info(f"WorldPop raster: {pop_raster_name}")


In [ ]:
# Downsample WorldPop (100 m, projected) onto native TerraClimate grid
# with total pop conserved -> SUM child pixels; nodata -> 0.
pop_aligned = np.full(ref_shape, 0.0, dtype=np.float64)
with rasterio.open(pop_raster_path) as pop_src:
    pop_nodata = pop_src.nodata
    log.info(
        f"  Source CRS: {pop_src.crs}, shape: {pop_src.width}x{pop_src.height}, "
        f"nodata: {pop_nodata}"
    )
    src_band = pop_src.read(1).astype(np.float64)
    # zero-out nodata BEFORE summing (constrained pop: nodata = no population)
    if pop_nodata is not None:
        src_band = np.where(src_band == pop_nodata, 0.0, src_band)
    src_band = np.where(np.isfinite(src_band), src_band, 0.0)

    reproject(
        source=src_band,
        destination=pop_aligned,
        src_transform=pop_src.transform,
        src_crs=pop_src.crs,
        dst_transform=ref_transform,
        dst_crs=ref_crs,                    # EPSG:4326 native grid
        resampling=Resampling.sum,
        src_nodata=0.0,
        dst_nodata=0.0,
    )

# Constrained pop convention: every cell now holds a summed count (>= 0).
# "has data" = received any population (or lies within populated extent).
pop_zero_filled = pop_aligned.astype(np.float32)
pop_with_data = pop_zero_filled > 0
log.info(
    f"  Pop downsampled (sum): {int((pop_zero_filled>0).sum()):,} non-empty cells, "
    f"total pop = {pop_zero_filled.sum():,.0f}"
)

##### 6. RASTERIZE ADMIN BOUNDARIES

In [ ]:
# Rasterize admin polygons onto the native grid (no reprojection: gdf is already
# EPSG:4326, same as ref_crs). all_touched=True is generous but pixels shared by
# several small admins are won by whichever is written last -> some tiny admins
# may still get zero pixels (handled by point-sampling in Section 7).

# sanity: native grid and gdf SAME crs
if ref_crs != gdf.crs:
    raise ValueError(
        f"CRS mismatch: native grid {ref_crs} != gdf {gdf.crs}. "
        f"Native grid and gdf CRS must match before rasterization."
    )

pcodes = sorted(gdf[pcode_col].tolist())
pcode_to_id = {p: i + 1 for i, p in enumerate(pcodes)}
id_to_pcode = {v: k for k, v in pcode_to_id.items()}

admin_shapes = [
    (geom, pcode_to_id[pcode])
    for geom, pcode in zip(gdf.geometry, gdf[pcode_col])
]
admin_raster = rasterize(
    admin_shapes, out_shape=ref_shape, transform=ref_transform,
    fill=0, dtype=np.int32, all_touched=True,
)

# inside-country all-NaN pixels — the only ones that mean anything for QC
inside_mask = admin_raster > 0
n_inside = int(inside_mask.sum())
n_inside_nodata = int((all_nan & inside_mask).sum())
log.info(
    f"Admin raster: {n_inside:,} pixels inside admin boundaries; "
    f"with data (all 12 months): {n_inside - n_inside_nodata:,}; "
    f"no data (all-NaN): {n_inside_nodata:,} "
    f"({100*n_inside_nodata/n_inside:.2f}%)"
)

# ── Identify admins that got NO pixel from areal rasterization ──
assigned_ids = set(np.unique(admin_raster)) - {0}
all_ids = set(pcode_to_id.values())
missing_ids = sorted(all_ids - assigned_ids)
missing_pcodes = [id_to_pcode[i] for i in missing_ids]
if missing_pcodes:
    log.warning(
        f"{len(missing_pcodes)} admin(s) received no areal pixel "
        f"(smaller than grid / lost pixel to neighbours): {missing_pcodes}"
    )


##### 7. COMPUTE ZONAL INDICATOR COMPONENTS
Same flat-array zonal aggregation as the flood pipeline, with dry-months in
place of flood-days.

In [ ]:
admin_ids_flat = admin_raster[inside_mask]
dry_flat = cumulative_dry_months[inside_mask]              # 0..12
pop_flat = pop_zero_filled[inside_mask]
pop_has_data_flat = pop_with_data[inside_mask]
dry_bool_flat = any_dry_mask[inside_mask].astype(bool)
valid_months_flat = valid_months[inside_mask]
all_nan_flat = all_nan[inside_mask]
area_flat = pixel_area_km2_grid[inside_mask]              # per-pixel km^2 (lat-dependent)

# Numerator (shared by pop-weighted indicators): dry-months x pop (person-months)
hp_flat = dry_flat * pop_flat
# Population where dry (indicator 1 numerator & indicator 3 denominator)
pop_where_dry_flat = np.where(dry_bool_flat, pop_flat, 0.0)

# Area contributions for the AREA indicators (now lat-weighted, not a constant)
dry_area_months_flat = dry_flat * area_flat              # km^2 * months per pixel
dry_pixel_area_flat = np.where(dry_bool_flat, area_flat, 0.0)  # area of dry pixels

pixel_df = pd.DataFrame({
    "admin_id": admin_ids_flat,
    "pop": pop_flat,
    "pop_where_dry": pop_where_dry_flat,
    "hp": hp_flat,
    "dry_months": dry_flat,
    "is_dry": dry_bool_flat.astype(int),
    "pop_has_data": pop_has_data_flat.astype(int),
    "pop_data_and_dry": (pop_has_data_flat & dry_bool_flat).astype(int),
    "valid_months": valid_months_flat,
    "is_all_nan": all_nan_flat.astype(int),
    "fully_observed": (valid_months_flat == 12).astype(int),
    "pixel_area_km2": area_flat,
    "dry_area_months_px": dry_area_months_flat,
    "dry_pixel_area_km2": dry_pixel_area_flat,
})
log.info(f"Pixel DataFrame: {len(pixel_df):,} rows")


In [ ]:
zonal = pixel_df.groupby("admin_id").agg(
    p_total=("pop", "sum"),
    p_dry=("pop_where_dry", "sum"),
    hp=("hp", "sum"),
    total_dry_months=("dry_months", "sum"),
    n_pixels=("pop", "count"),
    n_dry_pixels=("is_dry", "sum"),
    n_pop_data_pixels=("pop_has_data", "sum"),
    n_pop_data_and_dry=("pop_data_and_dry", "sum"),
    mean_valid_months=("valid_months", "mean"),
    min_valid_months=("valid_months", "min"),
    n_fully_observed=("fully_observed", "sum"),
    admin_area_km2=("pixel_area_km2", "sum"),          # total admin area (lat-weighted)
    dry_area_months_sum=("dry_area_months_px", "sum"), # sum of dry-months*area (km2*months)
    dry_area_km2=("dry_pixel_area_km2", "sum"),        # total dry area (km2)
).reset_index()

zonal[pcode_col] = zonal["admin_id"].map(id_to_pcode)
zonal["point_sampled"] = 0    # areal admins; point-sampled ones appended later
log.info(f"Zonal aggregation: {len(zonal)} admin areas")


In [ ]:
# ── POINT-SAMPLE admins that received no areal pixel ──
# For each admin with zero areal pixels, sample the single grid cell containing
# its polygon centroid (assumes that pixel is not all-NaN, otherwise fails).
# These rows are appended to `zonal` with point_sampled=1 so they are
# never silently mixed with areal aggregates. A point-sampled admin gets ONE
# pixel's worth of values; area indicators use that pixel's own area.

point_rows = []
for pc in missing_pcodes:
    admin_id = pcode_to_id[pc]
    geom = gdf.loc[gdf[pcode_col] == pc, "geometry"].iloc[0]
    cx, cy = geom.centroid.x, geom.centroid.y
    r, c = rowcol(ref_transform, cx, cy)
    H, W = ref_shape

    # sanity: r, c within bounds and not all-nan
    if not (0 <= r < H and 0 <= c < W):
        raise ValueError(
            f"Admin {pc} centroid outside grid ({r},{c})."
        )
    if all_nan[r, c]:
        raise ValueError(
            f"Admin {pc} centroid cell ({r},{c}) is all-NaN."
        )

    dm = int(cumulative_dry_months[r, c])
    vm = int(valid_months[r, c])
    is_nan = bool(all_nan[r, c])
    popv = float(pop_zero_filled[r, c])
    is_dry = bool(any_dry_mask[r, c])
    area = float(pixel_area_km2_grid[r, c])

    point_rows.append({
        "admin_id": admin_id,
        "p_total": popv,
        "p_dry": popv if is_dry else 0.0,
        "hp": dm * popv,
        "total_dry_months": dm,
        "n_pixels": 1,
        "n_dry_pixels": int(is_dry),
        "n_pop_data_pixels": int(popv > 0),
        "n_pop_data_and_dry": int((popv > 0) and is_dry),
        "mean_valid_months": vm,
        "min_valid_months": vm,
        "n_fully_observed": int(vm == 12),
        "admin_area_km2": area,
        "dry_area_months_sum": dm * area,
        "dry_area_km2": area if is_dry else 0.0,
        pcode_col: pc,
        "point_sampled": 1,
    })
    log.info(
        f"  Point-sampled {pc} at cell ({r},{c}): "
        f"dry_months={dm}, valid={vm}, pop={popv:.0f}"
    )

if point_rows:
    zonal = pd.concat([zonal, pd.DataFrame(point_rows)], ignore_index=True)
    log.info(f"Appended {len(point_rows)} point-sampled admin(s); zonal now {len(zonal)} rows")


In [ ]:
# ── POPULATION indicators (1-3) ── (unchanged: CRS-invariant ratios)
# 1. Impacted population rate (%): % of admin pop under any-month-dry mask
zonal["drought_impact_pop_pct"] = (zonal["p_dry"] / zonal["p_total"] * 100).round(2)
# 2. Pop-weighted avg dry-months (total-population denominator)
zonal["wavg_dry_months_total_pop"] = (zonal["hp"] / zonal["p_total"]).round(4)
# 3. Pop-weighted avg dry-months (intersection denominator: dry-affected pop)
zonal["wavg_dry_months_intersection"] = (zonal["hp"] / zonal["p_dry"]).round(4)

# ── AREA indicators (4-6) ── now LATITUDE-WEIGHTED (true km^2), not a constant
# 4. Dry area-months (territorial drought persistence, km^2 * months)
zonal["dry_area_months"] = zonal["dry_area_months_sum"].round(2)
# 5. Avg dry-months per total area  (sum(dry_months*area) / total area)
zonal["avg_dry_months_total_area"] = (
    zonal["dry_area_months_sum"] / zonal["admin_area_km2"]
).round(4)
# 6. Avg dry-months per dry area    (sum(dry_months*area) / dry area)
zonal["avg_dry_months_dry_area"] = (
    zonal["dry_area_months_sum"] / zonal["dry_area_km2"]
).round(4)

# ── QC KPIs ──
zonal["pct_area_dry"] = (zonal["dry_area_km2"] / zonal["admin_area_km2"] * 100).round(2)
zonal["pct_dry_with_pop"] = np.where(
    zonal["n_dry_pixels"] > 0,
    (zonal["n_pop_data_and_dry"] / zonal["n_dry_pixels"] * 100).round(2),
    np.nan,
)
# % of admin pixels that are fully observed (12 months)
zonal["pct_fully_observed"] = (zonal["n_fully_observed"] / zonal["n_pixels"] * 100).round(2)

# NOTE NaN handling: NOT propagated like flood; only the true-0/0 intersection case
# is zero-filled, and only for MEASURED admins (n_pixels>0, which now includes
# point-sampled admins that carry 1 pixel). Works for both pop and area indicators.
# Same for no admin area population (e.g. LBN admin3: small country, high admin level)
measured = zonal["n_pixels"] > 0
no_dry_pop = zonal["p_dry"] == 0
no_dry_area = zonal["dry_area_km2"] == 0
no_pop = zonal["p_total"] == 0
zonal.loc[measured & no_dry_pop, "wavg_dry_months_intersection"] = 0.0
zonal.loc[measured & no_dry_area, "avg_dry_months_dry_area"] = 0.0
zonal.loc[measured & no_pop, "drought_impact_pop_pct"]      = 0.0
zonal.loc[measured & no_pop, "wavg_dry_months_total_pop"]   = 0.0


In [ ]:
result_df = gdf[[pcode_col, name_col]].merge(
    zonal[[
        pcode_col, "p_total", "p_dry",
        "drought_impact_pop_pct", "wavg_dry_months_total_pop", "wavg_dry_months_intersection",
        "dry_area_months", "avg_dry_months_total_area", "avg_dry_months_dry_area",
        "n_pixels", "n_dry_pixels", "pct_area_dry",
        "n_pop_data_pixels", "n_pop_data_and_dry", "pct_dry_with_pop", "pct_fully_observed",
        "admin_area_km2", "dry_area_km2", "point_sampled",   # <-- carry area + flag
    ]],
    on=pcode_col, how="left",
).sort_values(pcode_col)

# Admins still absent from zonal (neither areal nor point-sampled) stay NaN.
# point_sampled: 1 = single-cell centroid sample, 0 = areal, NaN = truly no data.
n_point = int((result_df["point_sampled"] == 1).sum())
n_nodata = int(result_df["point_sampled"].isna().sum())
log.info(
    f"Result table: {len(result_df)} admin areas "
    f"({n_point} point-sampled, {n_nodata} still no-data)"
)

# Is point-sampling inflating areal admin-pop aggregation?
areal_pop = zonal.loc[zonal["point_sampled"] == 0, "p_total"].sum()
point_pop = zonal.loc[zonal["point_sampled"] == 1, "p_total"].sum()
log.info(
    f"areal admins total pop: {areal_pop:,.0f}, "
    f"point-sampled total pop: {point_pop:,.0f}, "
    f"true country pop: {pop_zero_filled.sum():,.0f}"
)

# Rate of point-sampled admins
log.info(
    f"point-sampled admins: {(zonal['point_sampled']==1).sum()} of {len(zonal)}"
)

# Ratio of population conserved (sum over inside pixels vs country)
pop_cons_ratio = round(result_df['p_total'].sum()/pop_aligned.sum(), 2)
log.info(
    f"result_df / country_pop: {pop_cons_ratio:,.0f}"
)

result_df.head()


##### 7b. EXPORT DRY-MONTHS × POP RASTER (person-months of drought)

In [ ]:
hp_raster = np.round(cumulative_dry_months * pop_zero_filled).astype(np.uint32)
log.info(f"Dry-months x persons raster: max={hp_raster.max():.0f} months x persons")
write_raster_cog(
    hp_raster, ref_transform, ref_crs,
    maps_hazards_folder + f"{drought_label}_{index_tag}_dry_months_x_pop.tif",
    dtype="uint32",
)

##### 8. BINARIZATION
Same helper as the flood pipeline: more dry-months = worse, so comparison is
always "above". Quantile classification included.

In [ ]:
def compute_binarization(series, comparison="gt", n_quantiles=4):
    """Binarize an indicator series: identify most-deprived admin areas.
    More drought = worse -> comparison="gt". NaN propagates.
    Mirrors compute_binarization in postprocess_gfm_flood.ipynb."""
    stat_mean = series.mean()
    stat_median = series.median()
    n_total = len(series)
    n_nodata = int(series.isna().sum())

    if comparison == "gt":
        prefix = "above"
        bin_mean = (series >= stat_mean).astype("Int64")
        bin_median = (series >= stat_median).astype("Int64")
    else:
        prefix = "below"
        bin_mean = (series <= stat_mean).astype("Int64")
        bin_median = (series <= stat_median).astype("Int64")

    nodata_mask = series.isna()
    bin_mean[nodata_mask] = pd.NA
    bin_median[nodata_mask] = pd.NA

    quant_name = f"quantiles_{n_quantiles}"
    quantile_codes = pd.qcut(series, q=n_quantiles, labels=False, duplicates="drop")
    n_quantiles_actual = int(quantile_codes.dropna().nunique())
    quantile_class = quantile_codes.map(
        lambda c: f"Q{int(c) + 1}" if pd.notna(c) else pd.NA
    ).astype(object)
    quantile_class[nodata_mask] = pd.NA

    stats = {
        "direction": prefix,
        "binarization_mean": round(stat_mean, 4) if pd.notna(stat_mean) else None,
        "binarization_median": round(stat_median, 4) if pd.notna(stat_median) else None,
        "n_regions_total": n_total,
        "n_regions_with_data": n_total - n_nodata,
        "n_regions_nodata": n_nodata,
        "n_deprived_by_mean": int(bin_mean.sum()) if not bin_mean.isna().all() else None,
        "n_deprived_by_median": int(bin_median.sum()) if not bin_median.isna().all() else None,
        "n_quantiles_requested": n_quantiles,
        "n_quantiles_actual": n_quantiles_actual,
    }
    return f"{prefix}_mean", bin_mean, f"{prefix}_median", bin_median, quant_name, quantile_class, stats

In [ ]:
bin_stats_all = {}

indicators = [
    ("drought_impact_pop_pct", "Impacted Pop %"),
    ("wavg_dry_months_total_pop", "Wavg Dry Months (total pop)"),
    ("wavg_dry_months_intersection", "Wavg Dry Months (intersection)"),
    ("dry_area_months", "Dry Area-Months"),
    ("avg_dry_months_total_area", "Avg Dry Months (total area)"),
    ("avg_dry_months_dry_area", "Avg Dry Months (dry area)"),
]

for col, label in indicators:
    mean_name, bin_m, med_name, bin_md, quant_name, quant_class, stats = compute_binarization(
        result_df[col], comparison="gt", n_quantiles=N_QUANTILES
    )
    result_df[f"{col}_{mean_name}"] = bin_m
    result_df[f"{col}_{med_name}"] = bin_md
    result_df[f"{col}_{quant_name}"] = quant_class
    bin_stats_all[col] = stats
    log.info(
        f"  {label}: mean={stats['binarization_mean']}, median={stats['binarization_median']}, "
        f"deprived(mean)={stats['n_deprived_by_mean']}, nodata={stats['n_regions_nodata']}"
    )
    if stats["n_quantiles_actual"] < stats["n_quantiles_requested"]:
        log.warning(
            f"    Quantiles requested={stats['n_quantiles_requested']}, "
            f"actual={stats['n_quantiles_actual']} (collapsed by tied values)"
        )

##### 9. EXCEL OUTPUT
One file per indicator. Population and area indicators carry the same sheet label.

In [ ]:
sheet_name = f"{c_iso3}_adm{admin_level}_{TARGET_YEAR}"

xlsx_filenames = [
    f"{drought_label}_{index_tag}_{ind[0]}.xlsx"
    for ind in indicators
]

output_cols = [pcode_col, name_col]
for col, _ in indicators:
    output_cols += [
        col, f"{col}_above_mean", f"{col}_above_median", f"{col}_quantiles_{N_QUANTILES}",
    ]

round_indicators = [2, 4, 4, 2, 4, 4]

for i in range(len(indicators)):
    xlsx_path = maps_hazards_folder + xlsx_filenames[i]
    result_df[
        output_cols[:2] + output_cols[2 + 4 * i:2 + 4 * (i + 1)]
    ].round(
        {indicators[i][0]: round_indicators[i]}
    ).astype(object).fillna("no-data").to_excel(
        xlsx_path, index=False, sheet_name=sheet_name,
    )
    log.info(f"Excel saved: {xlsx_path}")

##### 10a. CHOROPLETH MAPS (per indicator)

In [ ]:
def save_map_png(df, geojson_dict, pcode_col, indicator_col, title_text, out_path, color_scale="YlOrRd"):
    """Choropleth with grey nodata layer. Mirrors save_map_png in the flood pipeline."""
    has_nodata = df[indicator_col].isna().any()
    data_df = df.dropna(subset=[indicator_col])
    fig = go.Figure()
    if has_nodata:
        nodata_df = df[df[indicator_col].isna()]
        fig.add_trace(go.Choropleth(
            geojson=geojson_dict, featureidkey=f"properties.{pcode_col}",
            locations=nodata_df[pcode_col], z=[0] * len(nodata_df),
            colorscale=[[0, NODATA_GREY], [1, NODATA_GREY]], showscale=False,
            hoverinfo="location+text", text=["No data"] * len(nodata_df),
            marker_line_color="Gainsboro", marker_line_width=0.5,
        ))
    fig.add_trace(go.Choropleth(
        geojson=geojson_dict, featureidkey=f"properties.{pcode_col}",
        locations=data_df[pcode_col], z=data_df[indicator_col],
        colorscale=color_scale, colorbar=dict(len=0.65),
        marker_line_color="Gainsboro", marker_line_width=0.5,
    ))
    fig.update_geos(fitbounds="locations", visible=False, projection_type="mercator")
    fig.update_layout(
        title=dict(text=title_text, x=0.5, y=0.97),
        margin={"r": 0, "t": 0, "l": 0, "b": 0}, width=1200, height=900,
    )
    fig.write_image(out_path, format="png", scale=2)

In [ ]:
pio.renderers.default = None

map_configs = []
for i, (col, label) in enumerate(indicators):
    lbl = drought_label
    file_suffix = f"{lbl}_{index_tag}_{col}"
    title = f"{label} — {index_tag} (dry<={DRY_THRESHOLD}) — {c_iso3} adm{admin_level} {TARGET_YEAR}"
    save_map_png(
        result_df, geojson_dict, pcode_col, col, title,
        maps_hazards_folder + f"{file_suffix}.png", color_scale=CSCALE_DROUGHT,
    )
    map_configs.append({"file_suffix": file_suffix, "indicator": col})
    log.info(f"  Map saved: {file_suffix}.png")

##### 10b. CUMULATIVE DRY-MONTHS & DRY-MONTHS×POP OVERLAY MAPS
Because the drought source is already coarse (~TerraClimate resolution), there is
no need for viz downsampling or adaptive marker sizing. We
plot each actual reference-grid pixel as a fixed-size square marker (sized to
roughly match the pixel footprint on screen), coloured by value. If maps look too
sparse/dense once you test them, adjust `SQUARE_MARKER_SIZE` — the downsampling
machinery can be reintroduced later if ever needed.

In [ ]:
# Fixed marker size (px) for the square pixel overlay. Tune when testing look.
SQUARE_MARKER_SIZE = 3

def ref_pixels_to_4326(grid, ref_transform, ref_crs, threshold=0):
    """Return (lons, lats, values) of reference-grid cells with value > threshold,
    with pixel-centre coordinates reprojected to EPSG:4326 for plotting.

    No downsampling: every qualifying native pixel is emitted as one point.
    """
    rows, cols = np.where(grid > threshold)
    if len(rows) == 0:
        return [], [], np.array([])
    # pixel-centre coords in the UTM ref CRS
    xs, ys = xy(ref_transform, rows, cols)   # returns lists in ref_crs metres
    # reproject the point coordinates UTM -> 4326
    transformer = pyproj.Transformer.from_crs(ref_crs, "EPSG:4326", always_xy=True)
    lons, lats = transformer.transform(np.asarray(xs), np.asarray(ys))
    return list(lons), list(lats), grid[rows, cols]


def save_square_overlay_map(gdf, geojson_dict, pcode_col, lons, lats, values,
                            colorscale, colorbar_title, title_text, out_path,
                            marker_size=SQUARE_MARKER_SIZE):
    """Admin basemap + fixed-size opaque SQUARE markers, one per native pixel."""
    fig = go.Figure()
    # admin basemap
    fig.add_trace(go.Choropleth(
        geojson=geojson_dict, featureidkey=f"properties.{pcode_col}",
        locations=gdf[pcode_col], z=[0] * len(gdf),
        colorscale=[[0, "white"], [1, "white"]], showscale=False,
        marker_line_color="DarkGrey", marker_line_width=1, hoverinfo="skip",
    ))
    # pixel squares (fixed size, opaque, no rank scaling)
    fig.add_trace(go.Scattergeo(
        lon=list(lons), lat=list(lats), mode="markers",
        marker=dict(
            size=marker_size, symbol="square",
            color=list(values), colorscale=colorscale,
            showscale=True, colorbar=dict(title=colorbar_title, len=0.6),
            opacity=0.5,
        ),
        name=title_text, hoverinfo="name",
    ))
    fig.update_geos(fitbounds="locations", visible=False,
                    projection_type="mercator", bgcolor="white")
    fig.update_layout(
        title=dict(text=title_text, x=0.5, y=0.97),
        margin={"r": 0, "t": 30, "l": 0, "b": 0}, width=1200, height=900,
        showlegend=False,
    )
    fig.write_image(out_path, format="png", scale=2)

In [ ]:
# Cumulative dry-months overlay (one square per native pixel)
# NOTE: given native grid, reprojection not needed; serves only for thresholding
lons, lats, vals = ref_pixels_to_4326(cumulative_dry_months, ref_transform, ref_crs, threshold=0)
if len(vals):
    save_square_overlay_map(
        gdf, geojson_dict, pcode_col, lons, lats, vals,
        CSCALE_CUM_RASTER, "dry months",
        f"Cumulative Dry Months — {index_tag} — {drought_label}",
        maps_hazards_folder + f"{drought_label}_{index_tag}_cumulative_dry_months_map.png",
    )
    log.info("  Cumulative dry-months overlay map saved")

# Dry-months x pop overlay
# NOTE: given native grid, reprojection not needed; serves only for thresholding
lons2, lats2, vals2 = ref_pixels_to_4326(hp_raster, ref_transform, ref_crs, threshold=0)
if len(vals2):
    save_square_overlay_map(
        gdf, geojson_dict, pcode_col, lons2, lats2, vals2,
        CSCALE_RASTER, "months x persons",
        f"Dry Months x Pop — {index_tag} — {drought_label}",
        maps_hazards_folder + f"{drought_label}_{index_tag}_dry_months_x_pop_map.png",
    )
    log.info("  Dry-months x pop overlay map saved")

##### 10c. MONTHLY SPEI EVOLUTION — subplot grid + animated HTML
Continuous per-month SPEI (no thresholding), on a diverging colour scale with a SHARED range across all 12 months so panels/frames are directly comparable. The range hugs the year's actual data (zmin/zmax = country min/max over all months) while zmid=0 keeps zero at the neutral colour — preserving discrimination even when the year is lopsided wet or dry. Two artifacts from ONE extraction: a 4×3 quarter-row subplot grid (PNG) and an animated HTML (month slider + play).

In [ ]:
# plot spei helpers

def extract_month_points(tif_path):
    """Return (lons, lats, vals) for EVERY observed pixel (no thresholding).
    Native grid is already EPSG:4326, so pixel centres need no reprojection."""
    with rasterio.open(tif_path) as src:
        band = src.read(1).astype(np.float32)
        nod = src.nodata
        tf = src.transform
    if nod is not None and not np.isnan(nod):
        band = np.where(band == nod, np.nan, band)
    rows, cols = np.where(~np.isnan(band))
    if len(rows) == 0:
        return [], [], np.array([])
    xs, ys = xy(tf, rows, cols)   # lon, lat directly (native 4326)
    return list(xs), list(ys), band[rows, cols]

# Build the per-month point sets once and derive the shared colour range.
month_points = []          # list of (label, lons, lats, vals)
_all_vals = []
for month_label, tif_path in index_tifs:
    lons, lats, vals = extract_month_points(tif_path)
    month_points.append((month_label, lons, lats, vals))
    if len(vals):
        _all_vals.append(vals)

_all_vals = np.concatenate(_all_vals) if _all_vals else np.array([0.0])
_dmin = float(_all_vals.min())
_dmax = float(_all_vals.max())

# Color scale evaluation, try always red = dry, green = wet, yellow = transition
SPEI_MARKER_SIZE = SQUARE_MARKER_SIZE * 1.5

if _dmin < 0 < _dmax:
    # Data straddles zero -> diverging, symmetric about zero so green/red split
    # sits at the real wet/dry boundary. Symmetric bound = farthest extreme.
    _lim = max(abs(_dmin), abs(_dmax))
    SPEI_ZMIN = -float(np.ceil(_lim * 10) / 10)
    SPEI_ZMAX =  float(np.ceil(_lim * 10) / 10)
    SPEI_ZMID = 0.0
    SPEI_COLORSCALE = "RdYlGn"
    SPEI_REVERSESCALE = False          # red=low=dry, green=high=wet
elif _dmax <= 0:
    # Entirely dry all year -> sequential yellow->red over the actual dry range.
    SPEI_ZMIN = float(np.floor(_dmin * 10) / 10)
    SPEI_ZMAX = float(np.ceil(_dmax * 10) / 10)
    SPEI_ZMID = None                   # no midpoint anchoring
    SPEI_COLORSCALE = "YlOrRd"         # yellow(mild) -> red(severe)
    SPEI_REVERSESCALE = True           # low(most negative)=red, high=yellow
else:  # _dmin >= 0
    # Entirely wet all year -> sequential yellow->green.
    SPEI_ZMIN = float(np.floor(_dmin * 10) / 10)
    SPEI_ZMAX = float(np.ceil(_dmax * 10) / 10)
    SPEI_ZMID = None
    SPEI_COLORSCALE = "YlGn"           # yellow(mild) -> green(wettest)
    SPEI_REVERSESCALE = False

log.info(
    f"SPEI evolution colour range (shared): zmin={SPEI_ZMIN}, zmax={SPEI_ZMAX}, "
    f"zmid={SPEI_ZMID} (over {_all_vals.size:,} observed pixel-months)"
)


In [ ]:
# ── One routine, two artifacts: the 4×3 quarter-row subplot grid (PNG) and the
#    animated HTML — both from `month_points` and the shared SPEI_Z* range. ──

# Layout: 4 rows (quarters) x 3 cols (months within quarter). Flip to (3,4) for
# wide slides by swapping GRID_ROWS/GRID_COLS.
GRID_ROWS, GRID_COLS = 4, 3
_MONTH_NAMES = ["Jan","Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"]

def _month_title(label):
    # label is "YYYY-MM"
    m = int(label.split("-")[1])
    return _MONTH_NAMES[m - 1]

def _spei_marker(vals, show_scale):
    m = dict(
        size=SPEI_MARKER_SIZE, symbol="square",
        color=list(vals), colorscale=SPEI_COLORSCALE, reversescale=SPEI_REVERSESCALE,
        cmin=SPEI_ZMIN, cmax=SPEI_ZMAX,
        showscale=show_scale,
        colorbar=dict(title="SPEI", len=0.6) if show_scale else None,
        opacity=0.75,
    )
    if SPEI_ZMID is not None:
        # only anchor midpoint in the diverging case
        m["cmid"] = SPEI_ZMID
    return m


In [ ]:
def save_spei_grid(out_path):
    """4×3 (quarter-row) subplot grid of monthly SPEI, shared diverging scale."""
    fig = make_subplots(
        rows=GRID_ROWS, cols=GRID_COLS,
        specs=[[{"type": "scattergeo"}] * GRID_COLS for _ in range(GRID_ROWS)],
        subplot_titles=[_month_title(mp[0]) for mp in month_points],
        horizontal_spacing=0.01, vertical_spacing=0.03,
    )
    geo_ids = []
    for k, (label, lons, lats, vals) in enumerate(month_points):
        r, c = k // GRID_COLS + 1, k % GRID_COLS + 1
        # white admin basemap per panel
        fig.add_trace(go.Choropleth(
            geojson=geojson_dict, featureidkey=f"properties.{pcode_col}",
            locations=gdf[pcode_col], z=[0] * len(gdf),
            colorscale=[[0, "white"], [1, "white"]], showscale=False,
            marker_line_color="DarkGrey", marker_line_width=0.5, hoverinfo="skip",
        ), row=r, col=c)
        # colorbar only on the first panel (shared scale)
        fig.add_trace(go.Scattergeo(
            lon=lons, lat=lats, mode="markers",
            marker=_spei_marker(vals, show_scale=(k == 0)), hoverinfo="skip",
        ), row=r, col=c)
        geo_ids.append("geo" if k == 0 else f"geo{k+1}")

    # fit every panel to the country
    for gid in geo_ids:
        fig.layout[gid].update(fitbounds="locations", visible=False,
                               projection_type="mercator", bgcolor="white")
    fig.update_layout(
        title=dict(text=f"Monthly SPEI-{INDEX_SCALE} — {c_iso3} {TARGET_YEAR}",
                   x=0.5, y=0.99),
        margin={"r": 0, "t": 60, "l": 0, "b": 0},
        width=1200, height=1400, showlegend=False,
    )
    fig.write_image(out_path, format="png", scale=2)
    log.info(f"  SPEI grid saved: {out_path}")


In [ ]:
def save_spei_animation_html(out_path):
    """Animated HTML: one frame per month, shared scale, slider + play button."""
    label0, lons0, lats0, vals0 = month_points[0]
    base_admin = go.Choropleth(
        geojson=geojson_dict, featureidkey=f"properties.{pcode_col}",
        locations=gdf[pcode_col], z=[0] * len(gdf),
        colorscale=[[0, "white"], [1, "white"]], showscale=False,
        marker_line_color="DarkGrey", marker_line_width=1, hoverinfo="skip",
    )
    fig = go.Figure(
        data=[base_admin,
              go.Scattergeo(lon=lons0, lat=lats0, mode="markers",
                            marker=_spei_marker(vals0, show_scale=True), hoverinfo="skip")],
        frames=[
            go.Frame(
                name=_month_title(label),
                data=[base_admin,
                      go.Scattergeo(lon=lons, lat=lats, mode="markers",
                                    marker=_spei_marker(vals, show_scale=True),
                                    hoverinfo="skip")],
            )
            for (label, lons, lats, vals) in month_points
        ],
    )
    fig.update_geos(fitbounds="locations", visible=False,
                    projection_type="mercator", bgcolor="white")
    fig.update_layout(
        title=dict(text=f"Monthly SPEI-{INDEX_SCALE} — {c_iso3} {TARGET_YEAR}",
                   x=0.5, y=0.97),
        margin={"r": 0, "t": 40, "l": 0, "b": 0}, width=1000, height=800,
        showlegend=False,
        updatemenus=[dict(
            type="buttons", showactive=False, x=0.05, y=0.05, xanchor="left",
            buttons=[
                dict(label="▶ Play", method="animate",
                     args=[None, dict(frame=dict(duration=900, redraw=True),
                                      fromcurrent=True, transition=dict(duration=0))]),
                dict(label="❚❚ Pause", method="animate",
                     args=[[None], dict(frame=dict(duration=0, redraw=False),
                                        mode="immediate")]),
            ],
        )],
        sliders=[dict(
            active=0, x=0.15, y=0.02, len=0.8,
            currentvalue=dict(prefix="Month: "),
            steps=[dict(method="animate", label=_month_title(mp[0]),
                        args=[[_month_title(mp[0])],
                              dict(mode="immediate",
                                   frame=dict(duration=0, redraw=True),
                                   transition=dict(duration=0))])
                   for mp in month_points],
        )],
    )
    fig.write_html(out_path, include_plotlyjs="cdn", auto_play=False)
    log.info(f"  SPEI animation saved: {out_path}")


In [ ]:
# Produce both artifacts
save_spei_grid(
    maps_hazards_folder + f"{drought_label}_{index_tag}_spei_monthly_grid.png"
)
save_spei_animation_html(
    maps_hazards_folder + f"{drought_label}_{index_tag}_spei_monthly_animation.html"
)

##### 11. JSON RUN LOG

In [ ]:
admin_qc = []
for _, row in result_df.iterrows():
    admin_qc.append({
        "pcode": row[pcode_col],
        "name": row[name_col],
        "total_pixels": int(row["n_pixels"]) if pd.notna(row.get("n_pixels")) else 0,
        "dry_pixels": int(row["n_dry_pixels"]) if pd.notna(row.get("n_dry_pixels")) else 0,
        "pct_area_dry": float(row["pct_area_dry"]) if pd.notna(row.get("pct_area_dry")) else 0,
        "pop_data_pixels": int(row["n_pop_data_pixels"]) if pd.notna(row.get("n_pop_data_pixels")) else 0,
        "dry_pixels_with_pop_data": int(row["n_pop_data_and_dry"]) if pd.notna(row.get("n_pop_data_and_dry")) else 0,
        "pct_dry_with_pop_data": float(row["pct_dry_with_pop"]) if pd.notna(row.get("pct_dry_with_pop")) else None,
        "total_population": round(float(row["p_total"]), 1),
        "impacted_population": round(float(row["p_dry"]), 1),
        "drought_impact_pop_pct": float(row["drought_impact_pop_pct"]),
        "wavg_dry_months_total_pop": float(row["wavg_dry_months_total_pop"]),
        "wavg_dry_months_intersection": float(row["wavg_dry_months_intersection"]),
        "pct_fully_observed": round(float(row["pct_fully_observed"]), 2),
        "is_point_sampled": bool(row["point_sampled"]) if pd.notna(row.get("point_sampled")) else False,
    })

In [ ]:
run_log = {
    "pipeline": "postprocess_drought_index",
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "country": country,
    "iso3": c_iso3,
    "admin_level": admin_level,
    "population_year": year,
    "target_year": TARGET_YEAR,
    "index_kind": INDEX_KIND,
    "index_distribution": INDEX_DIST,
    "index_scale": INDEX_SCALE,
    "dry_threshold": DRY_THRESHOLD,
    "shapefile": geojson_file[0],
    "n_admin_regions": len(gdf),
    "population_raster": pop_raster_name,
    "index_raster_dir": str(drought_raster_dir),
    "reference_grid_source": "native_terraclimate_epsg4326",
    "grid_crs": str(ref_crs),
    "cell_size_deg": [float(cell_deg_x), float(cell_deg_y)],
    "grid_shape": list(ref_shape),
    "pixel_area_km2_range": [float(pixel_area_km2_grid.min()), float(pixel_area_km2_grid.max())],
    "months_stacked": months_loaded,
    "n_months": len(months_loaded),

    "dry_mask_method": "threshold on native grid",
    "pop_resampling": "sum",
    "admin_rasterize_all_touched": True,
    "pop_nodata_treatment": "zero_filled (constrained pop: nodata = no population)",
    "nan_handling": {
        "policy": "drought NaN carried, not treated as not-dry (unlike flood nodata->0)",
        "accumulation": "nansum: NaN months ignored; all-NaN pixel -> nodata",
        "cum_nodata_sentinel": CUM_NODATA,
        "min_valid_months_for_dry": MIN_VALID_MONTHS_FOR_DRY,
        "n_inside_nodata": n_inside_nodata,
    },

    "dry_mask": {"n_dry_pixels": n_dry_pixels},

    "total_pop_in_country": round(float(np.nansum(pop_aligned)), 1),
    "total_pop_in_admin_areas": round(float(result_df["p_total"].sum()), 1),
    "pop_in_admin_to_country_ratio": pop_cons_ratio,
    "is_admin_pop_conserved": abs(pop_cons_ratio - 1) < 0.1,
    "total_pop_impacted_by_drought": round(float(result_df["p_dry"].sum()), 1),

    "binarization": bin_stats_all,
    "admin_qc": admin_qc,

    "outputs": {
        "xlsx": [maps_hazards_folder + f for f in xlsx_filenames],
        "maps": [maps_hazards_folder + f"{cfg['file_suffix']}.png" for cfg in map_configs],
        "rasters": [
            maps_hazards_folder + f"{drought_label}_{index_tag}_cumulative_dry_months.tif",
            maps_hazards_folder + f"{drought_label}_{index_tag}_dry_months_x_pop.tif",
        ],
    },
}

log_path = output_log_dir + f"postprocess_drought_{drought_label}_{index_tag}.json"
with open(log_path, "w") as f:
    json.dump(run_log, f, indent=4, default=str)

log.info(f"Run log saved: {log_path}")
log.info("Post-processing complete.")

##### 12. Validation Drought NaNs cell
To be run for verification of a given pcode, e.g. using the log postpro output.

In [ ]:
# # --- choose an admin to inspect (by pcode or name substring) ---
# INSPECT_PCODE = "SO1202"        # e.g. "AF0620"  (Nazyan, 40% observed)
# INSPECT_NAME  = "Nazyan"        # or match by name substring; ignored if pcode set
# MAX_PIXELS_TO_PRINT = 10        # cap console output

# # --- Tip: run once with a partially-observed admin (e.g. Nazyan, 40% in AFG) and once
# #     with a fully-observed one (INSPECT_NAME='Kabul' e.g. in AFG) to contrast the patterns.

# # resolve the admin_id for the chosen admin
# if INSPECT_PCODE is not None:
#     sel = gdf[gdf[pcode_col] == INSPECT_PCODE]
# else:
#     sel = gdf[gdf[name_col].str.contains(INSPECT_NAME, case=False, na=False)]
# if len(sel) == 0:
#     raise ValueError("No admin matched — check INSPECT_PCODE / INSPECT_NAME")
# sel_pcode = sel.iloc[0][pcode_col]
# sel_name  = sel.iloc[0][name_col]
# sel_admin_id = pcode_to_id[sel_pcode]
# print(f"Inspecting admin: {sel_pcode}  {sel_name}  (admin_id={sel_admin_id})")

# # reconstruct the per-month SPEI stack ON THE REFERENCE GRID, keeping real values
# # (not just the dry mask) so we can see the actual index and its NaNs.
# month_labels = [ml for ml, _ in index_tifs]
# n_months = len(index_tifs)
# spei_stack = np.full((n_months,) + ref_shape, np.nan, dtype=np.float32)

# for k, (month_label, tif_path) in enumerate(index_tifs):
#     with rasterio.open(tif_path) as src:
#         src_data = src.read(1).astype(np.float32)
#         src_nodata = src.nodata
#         src_transform = src.transform
#         src_crs = src.crs
#     # normalize nodata to NaN on the native grid
#     if src_nodata is not None and not np.isnan(src_nodata):
#         src_data = np.where(src_data == src_nodata, np.nan, src_data)
#     # reproject the CONTINUOUS index to the ref grid (nearest, to match the mask logic)
#     band = np.full(ref_shape, np.nan, dtype=np.float32)
#     reproject(
#         source=src_data, destination=band,
#         src_transform=src_transform, src_crs=src_crs,
#         dst_transform=ref_transform, dst_crs=ref_crs,
#         resampling=Resampling.nearest,
#         src_nodata=np.nan, dst_nodata=np.nan,
#     )
#     spei_stack[k] = band

# # pixels belonging to the chosen admin
# admin_pixel_mask = (admin_raster == sel_admin_id)
# rows, cols = np.where(admin_pixel_mask)
# n_admin_px = len(rows)
# print(f"Pixels inside this admin: {n_admin_px}")
# if n_admin_px == 0:
#     print("  (zero pixels — admin smaller than one output cell; nothing to show)")
# else:
#     # summarize validity for this admin
#     vm = valid_months[admin_pixel_mask]
#     print(f"  valid_months: min={vm.min()}, max={vm.max()}, mean={vm.mean():.2f}")
#     print(f"  fully-observed pixels (12 mo): {(vm==12).sum()}/{n_admin_px}")
#     print(f"  all-NaN pixels: {int((vm==0).sum())}")
#     print()

#     # print per-pixel monthly SPEI sequences (NaN shown as '   nan')
#     hdr = "  ".join(f"{ml[-2:]}" for ml in month_labels)   # month numbers
#     print(f"{'row,col':>12} {'valid':>5} {'dry':>4}   {hdr}")
#     order = np.argsort(vm)   # show least-observed first
#     for idx in order[:MAX_PIXELS_TO_PRINT]:
#         r, c = rows[idx], cols[idx]
#         seq = spei_stack[:, r, c]
#         n_valid = int(np.sum(~np.isnan(seq)))
#         n_dry = int(np.nansum(seq <= DRY_THRESHOLD))
#         seq_str = "  ".join((f"{v:5.1f}" if not np.isnan(v) else "  nan") for v in seq)
#         print(f"  ({r:>3},{c:>3}) {n_valid:>5} {n_dry:>4}   {seq_str}")
#     if n_admin_px > MAX_PIXELS_TO_PRINT:
#         print(f"  ... ({n_admin_px - MAX_PIXELS_TO_PRINT} more pixels not shown)")

# # --- all-NaN pixels in this admin: their (row, col) -> (lon, lat) ---
# nan_idx = np.where(valid_months[admin_pixel_mask] == 0)[0]
# if len(nan_idx):
#     nan_rows = rows[nan_idx]
#     nan_cols = cols[nan_idx]
#     nan_lons, nan_lats = xy(ref_transform, nan_rows, nan_cols)   # native 4326: lon, lat
#     print(f"\n  all-NaN pixels ({len(nan_idx)}), (r,c)  lat, lon:")
#     for r, c, lon, lat in zip(nan_rows, nan_cols, nan_lons, nan_lats):
#         print(f"    ({r:>3},{c:>3})  {lat:.4f}, {lon:.4f}")
# else:
#     print("\n  all-NaN pixels: none")
